In [ ]:
# The script is simple; it is used to split a .csv file into smaller parts due to the limitation on Snowflake, which has a maximum file size of 250MB per file.

In [3]:
import os
import pandas as pd  

In [4]:
# Configuration
INPUT_CSV = "foreclosures_output.csv" 
OUTPUT_FOLDER = "split_outputs"
MAX_FILE_SIZE_MB = 100
CHUNK_SIZE = 100000  # Number of rows for initial chunk loading
# Create folder if it does not exist
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

In [5]:
def estimate_row_size(input_csv, sample_size):
    """Estimate the average row size in bytes."""
    sample = pd.read_csv(input_csv, nrows=sample_size, low_memory=False)
    temp_file = "temp_sample.csv"
    sample.to_csv(temp_file, index=False)
    size = os.path.getsize(temp_file) / sample_size  # Size per row in bytes
    os.remove(temp_file)
    return size

def split_csv(input_csv, output_folder, max_file_size_mb, sample_size):
    os.makedirs(output_folder, exist_ok=True)
    avg_row_size = estimate_row_size(input_csv, sample_size)
    max_file_size_bytes = max_file_size_mb * 1024 * 1024  # Convert MB to bytes
    estimated_chunk_size = int(max_file_size_bytes / avg_row_size)
    
    chunk_list = []
    file_counter = 1
    
    for chunk in pd.read_csv(input_csv, chunksize=estimated_chunk_size, low_memory=False):
        chunk_list.append(chunk)
        temp_df = pd.concat(chunk_list)
        
        temp_file_path = os.path.join(output_folder, f"split_part_{file_counter}.csv")
        temp_df.to_csv(temp_file_path, index=False)
        file_size = os.path.getsize(temp_file_path)
        
        if file_size >= max_file_size_bytes * 0.98:  # Ensure file stays within limit
            print(f"Saved {temp_file_path} ({file_size / (1024 * 1024):.2f} MB)")
            file_counter += 1
            chunk_list = []  # Reset the list
        else:
            os.remove(temp_file_path)  # Delete temporary file
    
    if chunk_list:
        final_file_path = os.path.join(output_folder, f"split_part_{file_counter}.csv")
        pd.concat(chunk_list).to_csv(final_file_path, index=False)
        print(f"Saved {final_file_path}")


In [6]:
# Run the function
split_csv(INPUT_CSV, OUTPUT_FOLDER, MAX_FILE_SIZE_MB, CHUNK_SIZE)
print("Done!")


Saved split_outputs/split_part_1.csv (99.65 MB)
Saved split_outputs/split_part_2.csv (100.98 MB)
Saved split_outputs/split_part_3.csv (102.71 MB)
Saved split_outputs/split_part_4.csv (102.63 MB)
Saved split_outputs/split_part_5.csv (102.45 MB)
Saved split_outputs/split_part_6.csv
Done!
